# Conformal Prediction

Split conformal sobre las probabilidades de SEED (semilla 42): la validación calibra el umbral q̂ y el test se evalúa. Puntaje de no conformidad = 1 − P(clase verdadera); umbral = k-ésimo puntaje con k = ⌈(n+1)(1−α)⌉. Implementado en `src/eeg_uq/uncertainty/conformal.py` (sin librerías externas).

Conformal no cambia la detección (F1 = SEED) ni produce MI: su salida son conjuntos {fondo}, {huso}, {ambos} (ambiguo) o {} (vacío, abstención), con una garantía de cobertura **marginal** (promedio sobre todos los timesteps). Aquí se mide también la cobertura **por clase**, para ver si la garantía se cumple dentro de los husos.

Evaluaciones 0 y 1, cinco folds cada una; media ± desviación estándar por evaluación y para las dos juntas.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'configs' / 'default.yaml').exists()), Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from eeg_uq import report

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 250)


def show(figure):
    display(figure)
    plt.close(figure)

conformal = report.conformal_results()
COLUMNS = ['cobertura objetivo', 'cobertura total', 'cobertura husos', 'cobertura fondo', 'vacíos (abstención)', 'ambiguos', 'tamaño medio']

## 1. Cobertura y conjuntos por α

In [ ]:
for alpha in sorted(conformal['alpha'].unique()):
    print('alpha = {:.2f}'.format(alpha))
    display(report.by_evaluation(conformal[conformal['alpha'] == alpha], COLUMNS))

## 2. Cobertura total y por clase

Línea punteada: cobertura objetivo 1 − α. La cobertura total sigue al objetivo, pero la de los husos queda muy por debajo: la garantía la sostiene el fondo (~95% de los timesteps).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.6, 4.4), sharey=True, constrained_layout=True)
fig.suptitle('Conformal: cobertura empírica vs objetivo (promedio de folds; barra = ± desv. est.)')
for ax, (label, part) in zip(axes, [('Evaluación 0', conformal[conformal['evaluación'] == 0]), ('Evaluación 1', conformal[conformal['evaluación'] == 1]), ('Promedio eval 0 y 1', conformal)]):
    stats = part.groupby('cobertura objetivo')[['cobertura total', 'cobertura husos', 'cobertura fondo']].agg(['mean', 'std'])
    for column, color in (('cobertura total', '#2a78d6'), ('cobertura fondo', '#7fb3ec'), ('cobertura husos', '#e34948')):
        ax.errorbar(stats.index, stats[(column, 'mean')], yerr=stats[(column, 'std')], fmt='o-', capsize=3, color=color, label=column)
    ax.plot([0.75, 1.0], [0.75, 1.0], '--', color=report.CHANCE, lw=1, label='objetivo')
    ax.set(title=label, xlabel='Cobertura objetivo (1 − α)')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Cobertura empírica')
show(fig)

## 3. Cómo se expresa la incertidumbre

Con α pequeño aparecen conjuntos ambiguos (ambas clases); con α mayores, conjuntos vacíos (abstención).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.6, 4.4), sharey=True, constrained_layout=True)
fig.suptitle('Conformal: proporción de conjuntos vacíos y ambiguos (promedio de folds)')
for ax, (label, part) in zip(axes, [('Evaluación 0', conformal[conformal['evaluación'] == 0]), ('Evaluación 1', conformal[conformal['evaluación'] == 1]), ('Promedio eval 0 y 1', conformal)]):
    stats = part.groupby('alpha')[['vacíos (abstención)', 'ambiguos']].mean()
    x = range(len(stats))
    ax.bar([i - 0.2 for i in x], stats['vacíos (abstención)'], 0.4, color='#eb6834', label='vacíos (abstención)')
    ax.bar([i + 0.2 for i in x], stats['ambiguos'], 0.4, color='#4a3aa7', label='ambiguos')
    ax.set_xticks(list(x), ['α = {:.2f}'.format(a) for a in stats.index])
    ax.set(title=label)
    ax.grid(axis='y', alpha=0.25)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Proporción de timesteps')
show(fig)

## Hallazgos

Promedio de las dos evaluaciones (los resultados por evaluación son casi idénticos).

1. **La garantía marginal se cumple:** cobertura 0.990 / 0.949 / 0.899 / 0.799 para objetivos de 0.99 / 0.95 / 0.90 / 0.80.
2. **Pero la sostiene el fondo.** La cobertura en los husos es 0.87 / 0.56 / 0.30 / 0.03 para α = 0.01 / 0.05 / 0.10 / 0.20, mientras el fondo supera el objetivo. Con α = 0.05, en ~44% de los timesteps de huso el conjunto no contiene la respuesta correcta.
3. **La incertidumbre se expresa de dos formas:** con α = 0.01 aparecen conjuntos ambiguos (2.6%); con α ≥ 0.05, solo conjuntos vacíos o abstención (4.5% con α = 0.05).
4. **No cambia la detección** (F1 = SEED): Conformal reordena la probabilidad de SEED con un umbral, no agrega información.

**Trabajo futuro.** Conformal por clase (Mondrian), para garantizar la cobertura dentro de los husos y no solo en promedio.